In [1]:
import pandas as pd

# ROOT
root_path = "C:/Users/jackm/Documents/GitHub/ms-capstone-TTS-natlang-styleprompts"

# ParaSpeechCaps

In [54]:
from datasets import load_dataset
import pandas as pd

# Load the entire dataset
# dataset = load_dataset("ajd12342/paraspeechcaps")

# Load specific splits of the dataset
# train_scaled = load_dataset("ajd12342/paraspeechcaps", split="train_scaled") #Only has EMilia files
train_base = load_dataset("ajd12342/paraspeechcaps", split="train_base")
holdout = load_dataset("ajd12342/paraspeechcaps", split="holdout")
dev = load_dataset("ajd12342/paraspeechcaps", split="dev")
test = load_dataset("ajd12342/paraspeechcaps", split="test")

dfs = [
    # train_scaled.to_pandas(),
    train_base.to_pandas(),
    holdout.to_pandas(),
    dev.to_pandas(),
    test.to_pandas(),
]

# merge into one dataframe
df = pd.concat(dfs, ignore_index=True)

# filter out VoxCeleb
df = df[df["source"] == "expresso"]
df = df[df["relative_audio_path"].str.contains("conversational_vad_segmented")]

In [ ]:
df.head()

In [10]:
# Duration in hours
total_hours = df["duration"].sum() / 3600
print(total_hours)

23.082813877314813


In [ ]:
# visualize tag distribution

import pandas as pd
import numpy as np

def is_populated(x):
    if x is None:
        return False
    if isinstance(x, float) and np.isnan(x):
        return False
    if isinstance(x, str):
        return x != 'None' and x.strip() != ''
    if isinstance(x, (list, tuple, np.ndarray)):
        return len(x) > 0
    return True  # fallback for other types

df['has_intrinsic'] = df['intrinsic_tags'].apply(is_populated)
df['has_situational'] = df['situational_tags'].apply(is_populated)

def categorize(row):
    if row['has_intrinsic'] and row['has_situational']:
        return 'both'
    elif row['has_intrinsic']:
        return 'intrinsic_only'
    elif row['has_situational']:
        return 'situational_only'
    else:
        return 'neither'

df['tag_category'] = df.apply(categorize, axis=1)

counts = (
    df.groupby(['source', 'tag_category'])
      .size()
      .unstack(fill_value=0)
)

import matplotlib.pyplot as plt

counts.plot(kind='bar')

plt.title('Tag Presence by Source')
plt.xlabel('Source')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.legend(title='Category')
plt.tight_layout()

plt.show()

# StyleTalk

In [ ]:
df_styletalk_eval_meta = pd.read_csv(
    f"{root_path}/src/data/raw/styletalk/annotations/eval.csv",
    sep=",",            # delimiter
    encoding="utf-8",   # or "latin1"
    na_values=["NA", ""],
)

df_styletalk_eval_meta

,diag_id,context,curr_audio_id,res_audio_id,curr_text,res_text,curr_emotion,curr_speed,curr_volume,res_emotion,res_speed,res_volume
0,health_306,"A : I've been trying to eat healthier lately, ...",health_306/c_0.wav,health_306/r_0.wav,B: Sometimes it's hard to stick to those habit...,"Yeah, it can be a real challenge when life g...",unfriendly,fast,loud,unfriendly,normal,normal
1,health_306,"A : I've been trying to eat healthier lately, ...",health_306/c_2.wav,health_306/r_2.wav,B: Sometimes it's hard to stick to those habit...,"True, I'm hoping I can make these changes a ...",friendly,normal,normal,cheerful,normal,normal
2,music_478,"A : Oh, have you heard the new album by The We...",music_478/c_0.wav,music_478/r_0.wav,"B: Hmm, I might go to their concert next month.","Oh, that could be fun. If you're going, mayb...",neutral,normal,normal,neutral,normal,normal
3,movie_430,"A : Seriously, that last movie we saw was just...",movie_430/c_1.wav,movie_430/r_1.wav,B: I just hope the next one we pick doesn't en...,"Whoa, take it easy! We'll make sure it's a h...",unfriendly,fast,loud,unfriendly,normal,normal
4,movie_48,A : Did you see the latest superhero movie? It...,movie_48/c_2.wav,movie_48/r_2.wav,"B: I just prefer movies with more depth, you k...","Absolutely, there's plenty of films with dep...",friendly,normal,loud,cheerful,normal,normal
...,...,...,...,...,...,...,...,...,...,...,...,...
853,game_408,"A : I can't believe we aced that level, that w...",game_408/c_0.wav,game_408/r_0.wav,"B: Sure, just give me a second to set things u...","No worries, take your time, we're not in a r...",neutral,slow,quiet,friendly,normal,normal
854,food_233,A : I can't believe how amazing that new Thai ...,food_233/c_0.wav,food_233/r_0.wav,B: They did seem to have a lot of options on t...,"Exactly! There's so much to try, maybe anoth...",neutral,normal,normal,cheerful,normal,normal
855,music_21,"A : I just found this amazing new band, and th...",music_21/c_2.wav,music_21/r_2.wav,B: I think I've heard about them once before.,That's cool! They've been gaining popularity...,friendly,normal,normal,cheerful,normal,normal
856,music_21,"A : I just found this amazing new band, and th...",music_21/c_1.wav,music_21/r_1.wav,B: I think I've heard about them once before.,"Wait, why do you sound mad? Did you not like...",unfriendly,fast,loud,unfriendly,normal,loud


# LibriTTS-P

In [3]:


df_metadatatags = pd.read_csv(
    f"{root_path}/src/data/raw/libritts/libritts-p/data/metadata_w_style_prompt_tags_v230922.csv",
    sep=",",            # delimiter
    encoding="utf-8",   # or "latin1"
    na_values=["NA", ""],
)

df_metadatatags


,item_name,spk_id,gender,pitch,speaking_speed,energy,content_prompt,style_prompt_key,raw_f0_mean,raw_f0_scale,raw_lf0_mean,raw_lf0_scale,raw_speaking_rate,raw_loudness_lufs,raw_loudness_mean,raw_loudness_scale,invalid
0,1001_134708_000013_000000,1001,M,low,very slow,low,"For man of you, your characteristic race, Here...",M_p-low_s-slow_e-low,110.46,32.68,4.66,0.28,3.68,-21.41,-2.92,3.62,0
1,1006_135212_000001_000004,1006,F,normal,normal,normal,Before laying it before the public it would be...,F_p-normal_s-normal_e-normal,214.32,28.74,5.36,0.13,5.14,-15.94,-1.24,3.05,0
2,1006_135212_000001_000005,1006,F,normal,normal,normal,These facts were briefly as follows:,F_p-normal_s-normal_e-normal,206.13,30.82,5.32,0.14,5.21,-17.26,-1.25,2.84,0
3,1006_135212_000002_000000,1006,F,high,normal,normal,At five o'clock on the evening of the eighteen...,F_p-high_s-normal_e-normal,225.86,34.47,5.41,0.14,4.92,-14.94,-1.65,3.43,0
4,1006_135212_000002_000001,1006,F,normal,normal,high,"It was a rainy, squally day, which grew wilder...",F_p-normal_s-normal_e-high,212.07,30.04,5.35,0.13,5.08,-15.34,-0.90,2.91,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373862,98_199_000029_000001,98,F,normal,normal,normal,"Every five minutes, by removing some of the cr...",F_p-normal_s-normal_e-normal,210.52,51.84,5.32,0.25,5.49,-19.72,-1.80,3.41,0
373863,98_199_000029_000002,98,F,normal,normal,high,She was now seen by many young men who had not...,F_p-normal_s-normal_e-high,217.92,60.05,5.35,0.27,4.80,-18.46,-0.81,2.56,0
373864,98_199_000029_000003,98,F,low,normal,low,"Not one, however, started with rapturous wonde...",F_p-low_s-normal_e-low,178.94,39.80,5.17,0.19,4.64,-20.00,-2.51,3.54,0
373865,98_199_000029_000004,98,F,normal,slow,low,"Yet Catherine was in very good looks, and had ...",F_p-normal_s-slow_e-low,220.13,46.15,5.37,0.21,4.47,-21.29,-2.49,3.43,0


In [4]:
df_prompt_cands = pd.read_csv(
    f"{root_path}/src/data/raw/libritts/libritts-p/data/style_prompt_candidates_v230922.csv",
    sep="|", 
    header=None
    , names=["style_prompt_key", "prompts"]
)

# split prompts to list
df_prompt_cands["prompts"] = df_prompt_cands["prompts"].str.split(";")

# explode for one propmt per row
df_prompt_cands = df_prompt_cands.assign(prompt=df_prompt_cands["prompts"]).explode("prompt")
df_prompt_cands["prompt"] = df_prompt_cands["prompt"].str.strip()
df_prompt_cands = df_prompt_cands.drop(columns="prompts")

df_prompt_cands

,style_prompt_key,prompt
0,M_p-low_s-slow_e-low,Ask a low-pitched man to speak slowly with low...
0,M_p-low_s-slow_e-low,Ask a man to speak slowly with low volume and ...
0,M_p-low_s-slow_e-low,"Ask a man to speak slowly, his pitch and volum..."
0,M_p-low_s-slow_e-low,"Ask a man with a low pitch to speak slowly, hi..."
0,M_p-low_s-slow_e-low,Command a man with a high-pitched voice to spe...
...,...,...
53,F_p-high_s-fast_e-high,"Let a woman speak with high pitch, fast speaki..."
53,F_p-high_s-fast_e-high,ask a high-pitched woman to speak fast with hi...
53,F_p-high_s-fast_e-high,ask a high-pitched woman to speak quckly with ...
53,F_p-high_s-fast_e-high,"ask a woman with a loud voice to say quickly, ..."


In [ ]:
# merge datasets, giving one prompt per audio file

merged = df_metadatatags.merge(
    df_prompt_cands,
    on="style_prompt_key",
    how="inner"
)
merged